---
execute:
  enabled: false
---

# Reto: ¿por qué preferimos MCO?

**MCO es insesgado. Pero hay muchos estimadores lineales e insesgados. ¿Por qué preferimos MCO?**

En el notebook anterior utilizamos simulación Monte Carlo para estudiar la insesgadez de los estimadores de MCO.

Ahora queremos estudiar otra propiedad: la **eficiencia**.

El teorema de Gauss-Markov establece que, dentro de la clase de estimadores lineales e insesgados, MCO tiene la menor varianza.

La pregunta es:

> **¿Podemos construir evidencia de esta propiedad mediante simulación?**


[Descargar este notebook (.ipynb)](https://github.com/amosino/mtx--courses/raw/refs/heads/main/MTX1/notebooks/05--reto_eficiencia_mco.ipynb)

## Dos estimadores de $\beta_2$

El estimador de MCO puede escribirse como

$$
b_2=\sum_{i=1}^{n} w_i y_i,
$$

donde

$$
w_i=
\frac{x_i-\bar x}
{\sum_{i=1}^{n}(x_i-\bar x)^2}.
$$

Consideremos ahora otro estimador lineal:

$$
\tilde\beta_2=\sum_{i=1}^{n} c_i y_i.
$$

Para que este estimador sea insesgado para $\beta_2$, los ponderadores deben satisfacer

$$
\sum_{i=1}^{n} c_i=0,
$$

y

$$
\sum_{i=1}^{n} c_i x_i=1.
$$


## Construimos los ponderadores de MCO

Usaremos nuevamente una muestra con `n = 20` observaciones y mantendremos fijos los valores de $x$.


In [ ]:
n <- 20
x <- 1:n

x_bar <- mean(x)

w <- (x - x_bar) / sum((x - x_bar)^2)


## Reto 1: construye un estimador alternativo

Construye un vector `c` que satisfaga las dos condiciones de insesgadez, pero que sea diferente de `w`.

### Pista algebraica

Una estrategia sencilla consiste en elegir libremente algunos ponderadores y utilizar las dos restricciones para determinar los dos restantes.

Por ejemplo, puedes fijar $c_1,\ldots,c_{n-2}$ y dejar $c_{n-1}$ y $c_n$ como incógnitas. Entonces deberás resolver

$$
c_{n-1}+c_n
=
-\sum_{i=1}^{n-2}c_i,
$$

y

$$
x_{n-1}c_{n-1}+x_nc_n
=
1-\sum_{i=1}^{n-2}c_ix_i.
$$

Tu vector `c` debe cumplir:

$$
\sum c_i=0,
\qquad
\sum c_ix_i=1,
$$

y debe ser diferente de los ponderadores de MCO.


In [ ]:
# Construye aquí los ponderadores alternativos

c <- 


## Verifica tu construcción

Antes de continuar, comprueba que tu estimador alternativo satisface las condiciones necesarias para ser insesgado.


In [ ]:
sum(c)
sum(c * x)
all.equal(c, w)


Si tu construcción es correcta, los dos primeros resultados deberían ser aproximadamente 0 y 1, respectivamente, y `all.equal(c, w)` debería indicar que ambos vectores son diferentes.


## Antes de simular: formula una predicción

Antes de ejecutar el experimento Monte Carlo, responde:

- ¿Esperas que $b_2$ y $\tilde\beta_2$ tengan aproximadamente la misma media?
- ¿Esperas que tengan la misma varianza?
- ¿Qué resultado sería consistente con el teorema de Gauss-Markov?


## Definimos el experimento Monte Carlo

Usaremos el mismo modelo del notebook anterior:

$$
y_i=\beta_1+\beta_2x_i+u_i,
$$

con

$$
\beta_1=0.2,\qquad
\beta_2=2,\qquad
\sigma^2=4.
$$

En cada repetición generaremos una nueva muestra de $y$ y calcularemos dos estimadores de $\beta_2$ usando exactamente la misma muestra.


In [ ]:
set.seed(123)

R <- 1000

beta_1 <- 0.2
beta_2 <- 2
sigma2 <- 4

y_sist <- beta_1 + beta_2 * x

b2_mco <- numeric(R)
b2_alt <- numeric(R)


## Reto 2: completa la simulación

En cada repetición debes:

1. generar un nuevo conjunto de errores $u_i$;
2. construir la nueva muestra de $y$;
3. calcular el estimador de MCO utilizando los ponderadores `w`;
4. calcular el estimador alternativo utilizando los ponderadores `c`.

Recuerda que ambos estimadores deben utilizar **la misma muestra** en cada repetición.


In [ ]:
for (i in 1:R) {

  # Genera una nueva muestra
  u <- 
  y <- 

  # Estimador de MCO
  b2_mco[i] <- 

  # Estimador alternativo
  b2_alt[i] <- 
}


## ¿Son ambos insesgados?

Compara los promedios Monte Carlo de ambos estimadores con el valor verdadero $\beta_2=2$.


In [ ]:
mean(b2_mco)
mean(b2_alt)

mean(b2_mco) - beta_2
mean(b2_alt) - beta_2


Si ambos estimadores son insesgados, sus distribuciones deberían estar aproximadamente centradas en el mismo valor verdadero.

Pero eso todavía no nos dice cuál estimador es más eficiente.


## Ahora comparemos la variabilidad

La eficiencia se refiere a la varianza del estimador bajo muestreo repetido.

Compara las varianzas Monte Carlo:


In [ ]:
var(b2_mco)
var(b2_alt)

var(b2_alt) / var(b2_mco)


La última expresión indica cuántas veces mayor es la varianza del estimador alternativo respecto de MCO.


## Observamos las distribuciones

Para que la comparación visual sea válida, utilizaremos los mismos límites horizontales en ambos histogramas.


In [ ]:
limites <- range(c(b2_mco, b2_alt))


### Distribución del estimador de MCO


In [ ]:
hist(b2_mco,
     freq = FALSE,
     xlim = limites,
     xlab = expression(b[2]),
     main = "")

lines(density(b2_mco))
abline(v = beta_2, lwd = 2)


### Distribución del estimador alternativo


In [ ]:
hist(b2_alt,
     freq = FALSE,
     xlim = limites,
     xlab = expression(tilde(beta)[2]),
     main = "")

lines(density(b2_alt))
abline(v = beta_2, lwd = 2)


## Resume la evidencia

Completa la siguiente tabla y utilízala para comparar ambos estimadores.


In [ ]:
resultados <- data.frame(
  Estimador = c("MCO", "Alternativo"),
  Media_MC = c(mean(b2_mco), mean(b2_alt)),
  Sesgo_MC = c(
    mean(b2_mco) - beta_2,
    mean(b2_alt) - beta_2
  ),
  Varianza_MC = c(
    var(b2_mco),
    var(b2_alt)
  )
)

resultados


## ¿Qué aprendiste?

Responde brevemente:

1. ¿Por qué no basta con comprobar que un estimador es insesgado?
2. ¿Cómo representaste computacionalmente la eficiencia?
3. ¿Qué papel tuvieron los ponderadores $w_i$ y $c_i$?
4. Si ambos estimadores son lineales e insesgados, ¿qué evidencia de la simulación explica por qué preferimos MCO?
5. ¿Qué resultado de este experimento es consistente con el teorema de Gauss-Markov?

> El reto no consiste solamente en ejecutar una simulación. Consiste en traducir una afirmación teórica sobre estimadores a un experimento computacional capaz de producir evidencia sobre ella.
